# Delinquency Dataset Preprocessing
## Solving Data Quality Issues

This notebook demonstrates how to clean and preprocess the delinquency dataset.

In [ ]:
# Install required packages (if needed)
# !pip install pandas openpyxl imbalanced-learn scikit-learn numpy

import pandas as pd
import numpy as np
from data_preprocessing import DelinquencyDataPreprocessor

## Option 1: Full Pipeline (Recommended)

In [ ]:
# Load and preprocess with one command
preprocessor = DelinquencyDataPreprocessor(
    "/Users/lexiloughlin/Desktop/Delinquency_prediction_dataset.xlsx"
)

clean_df = (preprocessor
            .run_full_pipeline(
                handle_imbalance=True,
                save_path="/Users/lexiloughlin/Desktop/Delinquency_CLEANED.xlsx"
            )
            .get_processed_data())

In [ ]:
# View cleaned data
print("\nCleaned Data:")
print(f"Shape: {clean_df.shape}")
print(f"\nColumns: {clean_df.columns.tolist()}")
print(f"\nFirst few rows:")
clean_df.head()

In [ ]:
# Check for missing values
print("Missing values after cleaning:")
missing = clean_df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None!")

## Option 2: Step-by-Step Control

In [ ]:
# For more control over each step
preprocessor = DelinquencyDataPreprocessor(
    "/Users/lexiloughlin/Desktop/Delinquency_prediction_dataset.xlsx"
)

# Step 1: Clean Employment Status
preprocessor.clean_employment_status()

In [ ]:
# Step 2: Handle missing values
preprocessor.handle_missing_income(method='employment_based')
preprocessor.handle_missing_loan_balance(method='median')
preprocessor.handle_missing_credit_score()

In [ ]:
# Step 3: Create derived features
preprocessor.flag_over_limit_customers()
preprocessor.engineer_payment_features()
preprocessor.identify_weak_features()

In [ ]:
# Step 4: Handle class imbalance
# Options: 'smote', 'oversample', or skip with handle_imbalance=False
preprocessor.balance_classes(method='smote')

In [ ]:
# Step 5: Get cleaned data
clean_df = preprocessor.get_processed_data()
preprocessor.create_quality_report()

# Save to file
preprocessor.save_processed_data(
    "/Users/lexiloughlin/Desktop/Delinquency_CLEANED.xlsx"
)

## Analysis: Before vs After

In [ ]:
# Compare original vs cleaned
original = pd.read_excel("/Users/lexiloughlin/Desktop/Delinquency_prediction_dataset.xlsx")
cleaned = pd.read_excel("/Users/lexiloughlin/Desktop/Delinquency_CLEANED.xlsx")

print("\n=" * 60)
print("BEFORE vs AFTER COMPARISON")
print("=" * 60)
print(f"\nOriginal:")
print(f"  Shape: {original.shape}")
print(f"  Missing values: {original.isnull().sum().sum()}")
print(f"  Delinquent rate: {original['Delinquent_Account'].mean()*100:.1f}%")

print(f"\nCleaned:")
print(f"  Shape: {cleaned.shape}")
print(f"  Missing values: {cleaned.isnull().sum().sum()}")
print(f"  Delinquent rate: {cleaned['Delinquent_Account'].mean()*100:.1f}%")
print(f"  New features: {set(cleaned.columns) - set(original.columns)}")

## Feature Importance Preview

In [ ]:
# Check correlations with target
correlation = clean_df.corr(numeric_only=True)['Delinquent_Account'].sort_values(ascending=False)
print("\nFeature Correlation with Delinquency:")
print(correlation[correlation.index != 'Delinquent_Account'])

## Ready for Modeling!

Your cleaned dataset is now ready for:
- Train/test split
- Model training (Logistic Regression, Random Forest, XGBoost, etc.)
- Cross-validation
- Hyperparameter tuning